In [ ]:
"""Điểm khởi đầu chính của ứng dụng."""

import os
import time
from datetime import datetime

sys.path.append('extract')

import config
import text_processor
import api_handler
import entity_processor
import output_manager
import utils
from api_handler import reset_request_counter



def process_entities() -> list:
    """Main function to process all files and extract entities."""
    # Reset counters
    reset_request_counter()
    
    all_entities = []
    
    file_contents = text_processor.read_files(config.INPUT_FILES)
    
    # Tính toán tổng số window trước để ước lượng
    total_windows = 0
    for file_path, content in file_contents.items():
        if not content:
            continue
        sentences = utils.split_sentences_vietnamese(content)
        windows = text_processor.create_non_overlapping_windows(sentences, window_size=config.WINDOW_SIZE)
        total_windows += len(windows)
    
    print(f"\nTotal estimated API requests (windows): {total_windows}")
    print(f"Estimated processing time: {total_windows * config.API_DELAY_SECONDS / 60:.1f} minutes (with rate limiting)")
    print("=" * 60)
    
    current_window = 0
    
    for file_path, content in file_contents.items():
        if not content:
            continue
            
        print(f"\nProcessing {file_path}...")
        
        sentences = utils.split_sentences_vietnamese(content)
        windows = text_processor.create_non_overlapping_windows(sentences, window_size=config.WINDOW_SIZE)
        
        for window in windows:
            current_window += 1
            print(f"\n[Progress: {current_window}/{total_windows} windows]")
            print(f"  Processing window {window['window_index'] + 1}/{len(windows)} of current file")
            
            entities = api_handler.extract_entities_with_gemini(window, file_path)
            
            for entity in entities:
                similar_entity = entity_processor.find_similar_entity(entity, all_entities)
                
                if similar_entity:
                    if entity_processor.merge_entities(similar_entity, entity):
                        print(f"    Merged entity: {entity['id']}")
                else:
                    all_entities.append(entity)
                    print(f"    New entity: {entity['id']} ({entity['type']})")
    
    # Post-process all entities
    all_entities = entity_processor.post_process_entities(all_entities)
    
    # Áp dụng làm sạch cuối cùng
    all_entities = entity_processor.cleanup_entities(all_entities)
    
    return all_entities
    
    return all_entities


def main():
    """Hàm main chính."""
    print("Starting entity extraction with improved logic...")
    
    # Cấu hình API
    import google.generativeai as genai
    genai.configure(api_key=config.GOOGLE_API_KEY)
    
    # Xử lý entities
    entities = process_entities()
    
    # Lưu kết quả
    output_manager.save_entities(entities)
    
    print("Entity extraction completed.")


if __name__ == "__main__":
    main()

Starting entity extraction with improved logic...

Total estimated API requests (windows): 189
Estimated processing time: 22.1 minutes (with rate limiting)

Processing SGK/Nguồn/Chủ đề 1/Bài 1.txt...

[Progress: 1/189 windows]
  Processing window 1/10 of current file
  [Request #1] Chủ đề 1 - Bài 1 - Window 1
  [Request #1] Completed - Extracted 3 entities
    New entity: Chiến tranh thế giới thứ hai (Sự kiện)
    New entity: phe Đồng minh chống phát xít (Khái niệm)
    New entity: Hội Quốc liên (Tổ chức)

[Progress: 2/189 windows]
  Processing window 2/10 of current file
  [Request #2] Chủ đề 1 - Bài 1 - Window 2
  [Request #2] Completed - Extracted 7 entities
    New entity: Liên Xô (Quốc gia)
    New entity: Anh (Quốc gia)
    New entity: Liên hợp quốc (Tổ chức)
    New entity: Tuyên bố về Liên hợp quốc (Văn kiện/Hiệp định)
    New entity: Hội nghị Tê-hê-ran (Hội nghị)
    New entity: Hội nghị I-an-ta (Hội nghị)
    New entity: Hiến chương Liên hợp quốc (Văn kiện/Hiệp định)

[Progre